# Global Program Analytics Dashboard

**Analyzing global program participation, academic alignment, student satisfaction, and program effectiveness for university-level global experience programs.**

This notebook implements the full analytics workflow described in the project README:

1. Data Collection
2. Data Cleaning
3. Data Integration
4. Exploratory Data Analysis (Program Demand, Student Satisfaction, Academic Alignment)
5. Survey Feedback Analysis
6. Predictive Modeling (Regression, Clustering, Forecasting)
7. SQL Analysis Layer
8. Export for Tableau Dashboard
9. Executive Summary & Key Insights

**Data note:** Program metadata (name, type, country, city, region, term, GPA/language requirements, etc.) is real public information collected from Northeastern University's Global Experience Office website (https://geo.northeastern.edu/), loaded from `GEO_programs_template_updated_31_programs.xlsx`. Student application records, enrollment outcomes, and survey feedback are **synthetically generated** for this project, since internal GEO student data is not publicly available. Synthetic data does not represent actual Northeastern student records.


## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import r2_score, silhouette_score

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100

RNG_SEED = 42
RNG = np.random.default_rng(RNG_SEED)

for d in ["data/raw", "data/processed"]:
    os.makedirs(d, exist_ok=True)

SOURCE_XLSX = "GEO_programs_template_updated_31_programs.xlsx"  # place this file in the same folder as the notebook

## 1. Data Collection

Load the real public GEO program metadata, then generate the synthetic datasets (applications, survey feedback, courses) that stand in for internal student records.


In [ ]:
programs_raw = pd.read_excel(SOURCE_XLSX, sheet_name="Programs")
programs_raw.to_csv("data/raw/geo_programs_raw.csv", index=False)

print(f"Loaded {len(programs_raw)} programs, {programs_raw.shape[1]} columns")
programs_raw.head()

## 2. Data Cleaning

- Standardize text fields and column names
- Coerce numeric fields (`min_gpa`)
- Fill missing categorical values with documented placeholders
- Drop duplicate program records
- Validate `program_type` against an expected set of categories
- Assign an `academic_area` and program `capacity` (synthetic — not part of the public GEO metadata, needed for academic-alignment and demand analysis)


In [ ]:
progs = programs_raw.copy()
progs.columns = [c.strip().lower() for c in progs.columns]

text_cols = ["program_name", "program_type", "country", "city", "region", "term"]
for col in text_cols:
    progs[col] = progs[col].astype(str).str.strip()

progs["language_requirement"] = progs["language_requirement"].fillna("Not specified")
progs["min_gpa"] = pd.to_numeric(progs["min_gpa"], errors="coerce")

before = len(progs)
progs = progs.drop_duplicates(subset="program_id").reset_index(drop=True)
print(f"Removed {before - len(progs)} duplicate program_id rows")

VALID_TYPES = {"Traditional Study Abroad", "Dialogue of Civilizations", "Summer In", "Exchange"}
invalid = set(progs["program_type"].unique()) - VALID_TYPES
assert not invalid, f"Unexpected program_type values: {invalid}"
print("program_type values validated:", sorted(progs['program_type'].unique()))

In [ ]:
# Synthetic enrichment: academic_area + capacity (not present in public GEO metadata)
ACADEMIC_AREAS = ["Business", "International Affairs", "Health Sciences", "Engineering",
                   "Arts & Humanities", "Social Sciences", "Computer Science", "Environmental Studies"]
AREA_WEIGHTS = [0.14, 0.16, 0.14, 0.12, 0.14, 0.14, 0.08, 0.08]

area_rng = np.random.default_rng(RNG_SEED + 5)
progs["academic_area"] = area_rng.choice(ACADEMIC_AREAS, size=len(progs), p=AREA_WEIGHTS)
progs["capacity"] = area_rng.integers(15, 40, size=len(progs))

progs.to_csv("data/processed/programs_cleaned.csv", index=False)
progs[["program_id", "program_name", "program_type", "region", "academic_area", "capacity"]].head()

### 2.1 Courses table (synthetic academic alignment detail)

In [ ]:
CREDIT_TYPES = ["NUpath Elective", "Major Requirement", "General Elective"]
NUPATH_OPTS = ["IC", "EI", "FQ", "AD", "DD", "WI", "ND"]

course_rows = []
cid = 1
for _, p in progs.iterrows():
    n_courses = int(area_rng.integers(3, 6))
    for _ in range(n_courses):
        course_rows.append({
            "course_id": cid,
            "program_id": p["program_id"],
            "course_title": f"{p['academic_area']} Topics {int(area_rng.integers(100, 499))}",
            "academic_area": p["academic_area"],
            "credit_type": area_rng.choice(CREDIT_TYPES),
            "nupath_attribute": area_rng.choice(NUPATH_OPTS),
        })
        cid += 1

courses = pd.DataFrame(course_rows)
courses.to_csv("data/processed/courses_cleaned.csv", index=False)
print(f"Generated {len(courses)} synthetic course records")
courses.head()

## 3. Synthetic Application & Enrollment Data

For each program, simulate five application cycles (`Fall 2024` → `Spring 2026`) with a per-program baseline demand and growth trend, so later sections can analyze demand trends and forecast future volume.


In [ ]:
TERMS_SEQ = ["Fall 2024", "Spring 2025", "Summer 2025", "Fall 2025", "Spring 2026"]
STATUSES = ["Accepted", "Rejected", "Withdrawn"]
STATUS_P = [0.72, 0.20, 0.08]

app_rows = []
aid = 1
sid_counter = 1000
for _, p in progs.iterrows():
    base_demand = int(area_rng.integers(10, 45))
    trend = area_rng.uniform(-0.05, 0.15)  # per-program growth/decline in demand across terms
    for t_idx, term in enumerate(TERMS_SEQ):
        n_apps = max(3, int(base_demand * (1 + trend) ** t_idx + area_rng.normal(0, 3)))
        for _ in range(n_apps):
            sid_counter += 1
            status = area_rng.choice(STATUSES, p=STATUS_P)
            enrolled = 1 if status == "Accepted" and area_rng.random() < 0.88 else 0
            if enrolled:
                completion = "Completed" if area_rng.random() < 0.94 else "In Progress"
            else:
                completion = "N/A"
            app_rows.append({
                "application_id": aid, "student_id": sid_counter, "program_id": p["program_id"],
                "application_term": term, "application_status": status,
                "enrolled": enrolled, "completion_status": completion,
            })
            aid += 1

applications = pd.DataFrame(app_rows)
applications.to_csv("data/raw/applications_raw.csv", index=False)
print(f"Generated {len(applications)} applications; overall enrollment rate = {applications['enrolled'].mean():.1%}")

In [ ]:
# Data quality checks
dup_apps = applications.duplicated(subset="application_id").sum()
orphan_apps = (~applications["program_id"].isin(progs["program_id"])).sum()
print(f"Duplicate application_ids: {dup_apps}")
print(f"Applications with no matching program: {orphan_apps}")

applications = applications.drop_duplicates(subset="application_id")
applications = applications[applications["program_id"].isin(progs["program_id"])]
applications.to_csv("data/processed/applications_cleaned.csv", index=False)

## 4. Synthetic Survey Feedback

Survey responses are generated only for **enrolled** students (with an 80% response rate). Scores are built from a per-student baseline plus noise, and `satisfaction_score` is modeled as a weighted function of academic fit, advising, and housing support — mirroring the real-world relationship the regression model in Section 8 will try to recover. Each response is also tagged with a feedback category and a representative feedback sentence.


In [ ]:
FEEDBACK_TEMPLATES = {
    "Academic Fit": ["Coursework matched my major well.", "Classes felt disconnected from my academic plan."],
    "Advising Support": ["My advisor was responsive and helpful.", "I had trouble reaching my program advisor."],
    "Housing & Logistics": ["Housing was comfortable and well located.", "Housing arrangements were confusing at first."],
    "Cost Concerns": ["The program was worth the cost.", "Program costs were higher than expected."],
    "Cultural Experience": ["I loved immersing in the local culture.", "Wish there were more cultural excursions."],
    "Program Communication": ["Communication from GEO was clear and timely.", "Updates about the program were inconsistent."],
    "Career Relevance": ["This experience will help my career goals.", "Not sure this connects to my career path."],
}
CATEGORIES = list(FEEDBACK_TEMPLATES.keys())

enrolled_apps = applications[applications["enrolled"] == 1].merge(
    progs[["program_id", "program_type", "region"]], on="program_id", how="left")

surv_rows = []
rid = 1
for _, a in enrolled_apps.iterrows():
    if area_rng.random() > 0.8:   # ~80% survey response rate
        continue
    base = area_rng.normal(4.0, 0.6)
    academic_fit = float(np.clip(base + area_rng.normal(0, 0.5), 1, 5))
    advisor_support = float(np.clip(base + area_rng.normal(0, 0.5), 1, 5))
    housing_support = float(np.clip(base + area_rng.normal(0, 0.5), 1, 5))
    satisfaction = float(np.clip(
        0.4 * academic_fit + 0.3 * advisor_support + 0.3 * housing_support + area_rng.normal(0, 0.3), 1, 5))
    recommend = float(np.clip(satisfaction + area_rng.normal(0, 0.3), 1, 5))

    cat = area_rng.choice(CATEGORIES)
    positive = satisfaction >= 3.5
    idx = 0 if positive else 1
    if area_rng.random() > 0.75:  # small chance of a mismatched-sentiment comment
        idx = 1 - idx
    feedback_text = FEEDBACK_TEMPLATES[cat][idx]

    surv_rows.append({
        "response_id": rid, "student_id": a["student_id"], "program_id": a["program_id"],
        "satisfaction_score": round(satisfaction, 2), "academic_fit_score": round(academic_fit, 2),
        "advisor_support_score": round(advisor_support, 2), "housing_support_score": round(housing_support, 2),
        "recommend_score": round(recommend, 2), "feedback_category": cat, "feedback_text": feedback_text,
    })
    rid += 1

survey = pd.DataFrame(surv_rows)
survey.to_csv("data/raw/survey_feedback_raw.csv", index=False)
print(f"Generated {len(survey)} survey responses")

In [ ]:
# Data quality checks: invalid score ranges, orphaned responses
bad_scores = ~survey["satisfaction_score"].between(1, 5)
orphan_survey = ~survey["program_id"].isin(progs["program_id"])
print(f"Out-of-range satisfaction scores: {bad_scores.sum()}")
print(f"Survey responses with no matching program: {orphan_survey.sum()}")

survey = survey[survey["satisfaction_score"].between(1, 5) & ~orphan_survey].reset_index(drop=True)
survey.to_csv("data/processed/survey_feedback_cleaned.csv", index=False)
survey.head()

## 5. Data Integration

Join program metadata with aggregated application/enrollment metrics and aggregated survey metrics into one analysis-ready table — `final_analysis_dataset` — the primary source for the EDA, modeling, and Tableau export.


In [ ]:
app_agg = applications.groupby("program_id").agg(
    total_applications=("application_id", "count"),
    total_enrolled=("enrolled", "sum"),
).reset_index()
app_agg["enrollment_rate"] = (app_agg["total_enrolled"] / app_agg["total_applications"]).round(3)

survey_agg = survey.groupby("program_id").agg(
    avg_satisfaction=("satisfaction_score", "mean"),
    avg_academic_fit=("academic_fit_score", "mean"),
    avg_advisor_support=("advisor_support_score", "mean"),
    avg_housing_support=("housing_support_score", "mean"),
    avg_recommend=("recommend_score", "mean"),
    n_responses=("response_id", "count"),
).reset_index().round(2)

final_dataset = (progs
                  .merge(app_agg, on="program_id", how="left")
                  .merge(survey_agg, on="program_id", how="left"))

final_dataset.to_csv("data/processed/final_analysis_dataset.csv", index=False)
print(final_dataset.shape)
final_dataset[["program_name", "region", "program_type", "total_applications",
               "enrollment_rate", "avg_satisfaction", "avg_academic_fit"]].head(10)

## 6. Exploratory Data Analysis — Program Demand

**Questions:** Which program types have the highest demand? Which regions attract the most interest? How does demand change across terms?


In [ ]:
demand_by_region = (applications.merge(progs[["program_id", "region"]], on="program_id")
                    .groupby("region")["application_id"].count()
                    .sort_values(ascending=False))
demand_by_type = (applications.merge(progs[["program_id", "program_type"]], on="program_id")
                   .groupby("program_type")["application_id"].count()
                   .sort_values(ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.barplot(x=demand_by_region.values, y=demand_by_region.index, ax=axes[0], color="#4C72B0")
axes[0].set_title("Total Applications by Region")
axes[0].set_xlabel("Applications")

sns.barplot(x=demand_by_type.values, y=demand_by_type.index, ax=axes[1], color="#55A868")
axes[1].set_title("Total Applications by Program Type")
axes[1].set_xlabel("Applications")
plt.tight_layout()
plt.show()

In [ ]:
apps_by_term = applications.groupby("application_term")["application_id"].count().reindex(TERMS_SEQ)

plt.figure(figsize=(8, 4.5))
sns.lineplot(x=apps_by_term.index, y=apps_by_term.values, marker="o", color="#C44E52")
plt.title("Application Volume Trend by Term")
plt.ylabel("Total Applications")
plt.xlabel("Term")
plt.tight_layout()
plt.show()

underfilled = final_dataset[final_dataset["enrollment_rate"] < final_dataset["enrollment_rate"].median()][
    ["program_name", "region", "enrollment_rate", "total_applications"]].sort_values("enrollment_rate").head(5)
print("Lowest enrollment-rate programs (candidates for review or reduced sections):")
underfilled

## 7. Exploratory Data Analysis — Student Satisfaction

**Questions:** Which programs/program types have the highest satisfaction? Is housing support associated with overall satisfaction? Do satisfaction levels differ by region?


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.boxplot(data=survey.merge(progs[["program_id", "program_type"]], on="program_id"),
            x="program_type", y="satisfaction_score", ax=axes[0], palette="Set2")
axes[0].set_title("Satisfaction Score by Program Type")
axes[0].tick_params(axis="x", rotation=20)

sns.scatterplot(data=final_dataset, x="enrollment_rate", y="avg_satisfaction",
                hue="region", ax=axes[1], s=70)
axes[1].set_title("Enrollment Rate vs. Average Satisfaction")
plt.tight_layout()
plt.show()

In [ ]:
corr = survey[["satisfaction_score", "academic_fit_score", "advisor_support_score", "housing_support_score",
                "recommend_score"]].corr()["satisfaction_score"].drop("satisfaction_score").sort_values(ascending=False)
print("Correlation of support ratings with overall satisfaction:")
corr

## 8. Academic Alignment Analysis

**Questions:** Which academic areas are most represented in global programs? Which areas have the strongest academic-fit ratings? Which academic areas are underserved by current offerings?


In [ ]:
area_counts = progs["academic_area"].value_counts()
area_fit = final_dataset.groupby("academic_area")["avg_academic_fit"].mean().round(2).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.barplot(x=area_counts.values, y=area_counts.index, ax=axes[0], color="#8172B2")
axes[0].set_title("Number of Programs by Academic Area")
sns.barplot(x=area_fit.values, y=area_fit.index, ax=axes[1], color="#CCB974")
axes[1].set_title("Average Academic Fit Score by Academic Area")
plt.tight_layout()
plt.show()

underserved = area_counts[area_counts <= area_counts.median()]
print("Academic areas with relatively few program offerings (opportunity areas):")
print(underserved)

## 9. Survey Feedback Analysis

Categorize open-ended feedback into recurring issue areas and see which categories skew negative (below-median satisfaction) most often, to flag where advising, housing, or communication improvements would help most.


In [ ]:
cat_counts = survey["feedback_category"].value_counts()

median_sat = survey["satisfaction_score"].median()
cat_sentiment = (survey.assign(below_median=survey["satisfaction_score"] < median_sat)
                  .groupby("feedback_category")["below_median"].mean().sort_values(ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.barplot(x=cat_counts.values, y=cat_counts.index, ax=axes[0], color="#64B5CD")
axes[0].set_title("Feedback Volume by Category")
sns.barplot(x=cat_sentiment.values, y=cat_sentiment.index, ax=axes[1], color="#DD8452")
axes[1].set_title("Share of Below-Median-Satisfaction Feedback\nby Category")
axes[1].set_xlabel("Share below median satisfaction")
plt.tight_layout()
plt.show()

print("Sample feedback comments:")
survey[["feedback_category", "feedback_text"]].drop_duplicates("feedback_category")

## 10. Predictive Modeling — Satisfaction Regression

Predict `satisfaction_score` from academic fit, advising support, housing support, program type, and region using linear regression, and inspect which factors matter most.


In [ ]:
model_df = survey.merge(progs[["program_id", "program_type", "region"]], on="program_id")

feature_cols = ["academic_fit_score", "advisor_support_score", "housing_support_score", "program_type", "region"]
X = pd.get_dummies(model_df[feature_cols], drop_first=True)
y = model_df["satisfaction_score"]

reg = LinearRegression().fit(X, y)
pred = reg.predict(X)

print(f"R² (in-sample): {r2_score(y, pred):.3f}")
coef_table = pd.Series(reg.coef_, index=X.columns).sort_values(key=abs, ascending=False)
coef_table.head(8).to_frame("coefficient")

In [ ]:
plt.figure(figsize=(6.5, 4.5))
sns.scatterplot(x=y, y=pred, alpha=0.4)
lims = [1, 5]
plt.plot(lims, lims, "--", color="gray")
plt.xlabel("Actual satisfaction score")
plt.ylabel("Predicted satisfaction score")
plt.title("Satisfaction Regression: Actual vs. Predicted")
plt.tight_layout()
plt.show()

## 11. Predictive Modeling — Program Performance Clustering

Group programs into performance profiles using enrollment rate, average satisfaction, and average recommend score with K-Means.


In [ ]:
cluster_features = final_dataset[["enrollment_rate", "avg_satisfaction", "avg_recommend"]].copy()
cluster_features = cluster_features.fillna(cluster_features.mean())

scaler = StandardScaler()
Xc = scaler.fit_transform(cluster_features)

km = KMeans(n_clusters=3, random_state=RNG_SEED, n_init=10).fit(Xc)
final_dataset["performance_cluster"] = km.labels_

sil = silhouette_score(Xc, km.labels_)
print(f"Silhouette score: {sil:.3f}")

cluster_profile = final_dataset.groupby("performance_cluster")[
    ["enrollment_rate", "avg_satisfaction", "avg_recommend"]].mean().round(3)
cluster_profile["n_programs"] = final_dataset["performance_cluster"].value_counts()
cluster_profile

In [ ]:
plt.figure(figsize=(6.5, 5))
sns.scatterplot(data=final_dataset, x="avg_satisfaction", y="enrollment_rate",
                hue="performance_cluster", palette="Set1", s=80)
plt.title("Program Performance Clusters")
plt.xlabel("Average Satisfaction")
plt.ylabel("Enrollment Rate")
plt.tight_layout()
plt.show()

CLUSTER_LABELS = cluster_profile["avg_satisfaction"].rank(ascending=False).map(
    {1.0: "High Performing", 2.0: "Moderate Performing", 3.0: "Needs Improvement"})
print(CLUSTER_LABELS)

## 12. Predictive Modeling — Demand Forecasting

Fit a simple linear trend on total applications per term to estimate next-term application volume, then break the same approach out by region.


In [ ]:
term_idx = {t: i for i, t in enumerate(TERMS_SEQ)}
apps_by_term = applications.groupby("application_term")["application_id"].count().reindex(TERMS_SEQ)

X_t = np.array([term_idx[t] for t in apps_by_term.index]).reshape(-1, 1)
y_t = apps_by_term.values

trend_model = LinearRegression().fit(X_t, y_t)
next_term_forecast = trend_model.predict([[len(TERMS_SEQ)]])[0]

print(f"Historical applications by term:\n{apps_by_term}\n")
print(f"Forecasted applications for the next term: {next_term_forecast:.0f}")

plt.figure(figsize=(8, 4.5))
plt.plot(list(apps_by_term.index) + ["Next Term"],
         list(apps_by_term.values) + [next_term_forecast],
         marker="o")
plt.axvline(x=len(TERMS_SEQ) - 0.5, linestyle="--", color="gray", alpha=0.6)
plt.title("Application Volume: Historical Trend + Forecast")
plt.ylabel("Total Applications")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
region_forecasts = {}
for region in progs["region"].unique():
    region_apps = (applications.merge(progs[["program_id", "region"]], on="program_id")
                    .query("region == @region")
                    .groupby("application_term")["application_id"].count()
                    .reindex(TERMS_SEQ).fillna(0))
    Xr = np.array([term_idx[t] for t in region_apps.index]).reshape(-1, 1)
    yr = region_apps.values
    m = LinearRegression().fit(Xr, yr)
    region_forecasts[region] = m.predict([[len(TERMS_SEQ)]])[0]

pd.Series(region_forecasts, name="forecasted_next_term_applications").sort_values(ascending=False).round(1)

## 13. SQL Analysis Layer

Load the cleaned tables into an in-memory SQLite database — mirroring the `programs`, `applications`, `survey_feedback`, and `courses` schema from the project's SQL design — and run representative analysis queries.


In [ ]:
conn = sqlite3.connect(":memory:")
progs.to_sql("programs", conn, index=False, if_exists="replace")
applications.to_sql("applications", conn, index=False, if_exists="replace")
survey.to_sql("survey_feedback", conn, index=False, if_exists="replace")
courses.to_sql("courses", conn, index=False, if_exists="replace")
print("Tables loaded:", pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)["name"].tolist())

In [ ]:
# Data quality check via SQL: applications with no matching program record
q_orphans = '''
SELECT COUNT(*) AS orphan_applications
FROM applications a
LEFT JOIN programs p ON a.program_id = p.program_id
WHERE p.program_id IS NULL;
'''
pd.read_sql(q_orphans, conn)

In [ ]:
# Demand by region and program type
q_demand = '''
SELECT p.region, p.program_type,
       COUNT(a.application_id) AS total_applications,
       SUM(a.enrolled) AS total_enrolled,
       ROUND(1.0 * SUM(a.enrolled) / COUNT(a.application_id), 3) AS enrollment_rate
FROM applications a
JOIN programs p ON a.program_id = p.program_id
GROUP BY p.region, p.program_type
ORDER BY total_applications DESC;
'''
pd.read_sql(q_demand, conn)

In [ ]:
# Average satisfaction and academic fit by academic area
q_satisfaction = '''
SELECT p.academic_area,
       ROUND(AVG(s.satisfaction_score), 2) AS avg_satisfaction,
       ROUND(AVG(s.academic_fit_score), 2) AS avg_academic_fit,
       COUNT(s.response_id) AS n_responses
FROM survey_feedback s
JOIN programs p ON s.program_id = p.program_id
GROUP BY p.academic_area
ORDER BY avg_satisfaction DESC;
'''
pd.read_sql(q_satisfaction, conn)

In [ ]:
conn.close()

## 14. Export for Tableau Dashboard

Write the final analysis-ready tables that a Tableau dashboard (Executive Overview, Program Demand, Student Satisfaction, Academic Alignment, Survey Feedback Insights, Predictive Planning pages) would connect to.


In [ ]:
final_dataset.to_csv("data/processed/final_analysis_dataset.csv", index=False)
applications.to_csv("data/processed/applications_cleaned.csv", index=False)
survey.to_csv("data/processed/survey_feedback_cleaned.csv", index=False)
courses.to_csv("data/processed/courses_cleaned.csv", index=False)

apps_by_term.to_frame("total_applications").to_csv("data/processed/applications_by_term.csv")

print("Exported files for Tableau:")
for f in sorted(os.listdir("data/processed")):
    print(" -", f)

## 15. Executive Summary — Key Insights

In [ ]:
top_region = demand_by_region.idxmax()
top_type = demand_by_type.idxmax()
best_area = area_fit.idxmax()
top_program = final_dataset.sort_values("avg_satisfaction", ascending=False).iloc[0]
worst_cat = cat_sentiment.idxmax()

print("KEY INSIGHTS")
print("-" * 60)
print(f"1. {top_region} generates the most application demand overall.")
print(f"2. {top_type} programs receive the highest total application volume.")
print(f"3. Housing and advising support ratings correlate positively with overall")
print(f"   satisfaction (see Section 7 correlation table).")
print(f"4. {best_area} programs show the strongest average academic fit ratings.")
print(f"5. Top-rated program by satisfaction: '{top_program['program_name']}' "
      f"({top_program['avg_satisfaction']:.2f}/5).")
print(f"6. '{worst_cat}' feedback skews most negative relative to other categories,")
print(f"   suggesting it's a priority area for improvement.")
print(f"7. Forecast: next-term application volume ~{next_term_forecast:.0f}, based on a")
print(f"   simple linear trend across the last {len(TERMS_SEQ)} terms.")
print(f"8. K-Means grouped programs into 3 performance profiles (silhouette={sil:.2f});")
print(f"   see Section 11 for programs flagged as 'Needs Improvement'.")

---
### Limitations
This project uses synthetic student-level data for demonstration purposes. Only the program metadata is real public information from Northeastern's GEO site. Results should not be interpreted as actual Northeastern University program performance or student outcomes.

### Future Improvements
- Connect the pipeline to live survey tools (Qualtrics, Google Forms)
- Automate dashboard refreshes and add API-based data ingestion
- Apply NLP to open-ended feedback text
- Build a recommendation system matching students to programs
- Expand analysis to cost, scholarships, and career outcomes
